# EDA and SQL

## Description of the datasets

- #### application_{train|test}.csv
This is the main table, broken into two files for Train (with TARGET) and Test (without TARGET).
Static data for all applications. One row represents one loan in our data sample.

- #### bureau.csv
All client's previous credits provided by other financial institutions that were reported to Credit Bureau (for clients who have a loan in our sample).
For every loan in our sample, there are as many rows as number of credits the client had in Credit Bureau before the application date.

- #### bureau_balance.csv
Monthly balances of previous credits in Credit Bureau.
This table has one row for each month of history of every previous credit reported to Credit Bureau – i.e the table has (#loans in sample * # of relative previous credits * # of months where we have some history observable for the previous credits) rows.

- #### POS_CASH_balance.csv
Monthly balance snapshots of previous POS (point of sales) and cash loans that the applicant had with Home Credit.
This table has one row for each month of history of every previous credit in Home Credit (consumer credit and cash loans) related to loans in our sample – i.e. the table has (#loans in sample * # of relative previous credits * # of months in which we have some history observable for the previous credits) rows.

- #### credit_card_balance.csv
Monthly balance snapshots of previous credit cards that the applicant has with Home Credit.
This table has one row for each month of history of every previous credit in Home Credit (consumer credit and cash loans) related to loans in our sample – i.e. the table has (#loans in sample * # of relative previous credit cards * # of months where we have some history observable for the previous credit card) rows.

- #### previous_application.csv
All previous applications for Home Credit loans of clients who have loans in our sample.
There is one row for each previous application related to loans in our data sample.

- #### installments_payments.csv
Repayment history for the previously disbursed credits in Home Credit related to the loans in our sample.
There is a) one row for every payment that was made plus b) one row each for missed payment.
One row is equivalent to one payment of one installment OR one installment corresponding to one payment of one previous Home Credit credit related to loans in our sample.

- #### HomeCredit_columns_description.csv
This file contains descriptions for the columns in the various data files.

## Imports

In [1]:
import duckdb

# Establish a connection to an in-memory DuckDB instance
con = duckdb.connect(database=':memory:')

# Register files as Views in DuckDB for easier querying later on
con.execute("CREATE VIEW IF NOT EXISTS app_train AS SELECT * FROM read_csv_auto('data/raw/application_train.csv')")
con.execute("CREATE VIEW IF NOT EXISTS bureau AS SELECT * FROM read_csv_auto('data/raw/bureau.csv')")
con.execute("CREATE VIEW IF NOT EXISTS bureau_balance AS SELECT * FROM read_csv_auto('data/raw/bureau_balance.csv')")
con.execute("CREATE VIEW IF NOT EXISTS credit_card AS SELECT * FROM read_csv_auto('data/raw/credit_card_balance.csv')")
con.execute("CREATE VIEW IF NOT EXISTS pos_cash AS SELECT * FROM read_csv_auto('data/raw/POS_CASH_balance.csv')")
con.execute("CREATE VIEW IF NOT EXISTS installments AS SELECT * FROM read_csv_auto('data/raw/installments_payments.csv')")
con.execute("CREATE VIEW IF NOT EXISTS prev_app AS SELECT * FROM read_csv_auto('data/raw/previous_application.csv')")

## SQL Queries

### Exploration

In [17]:
# Describe the structure of the CSV
schema_query = """
    DESCRIBE SELECT * FROM app_train;
"""

# Fetch and display schema
df_schema = con.execute(schema_query).df()
display(df_schema)

,column_name,column_type,null,key,default,extra
0,SK_ID_CURR,BIGINT,YES,None,None,None
1,TARGET,BIGINT,YES,None,None,None
2,NAME_CONTRACT_TYPE,VARCHAR,YES,None,None,None
3,CODE_GENDER,VARCHAR,YES,None,None,None
4,FLAG_OWN_CAR,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
117,AMT_REQ_CREDIT_BUREAU_DAY,DOUBLE,YES,None,None,None
118,AMT_REQ_CREDIT_BUREAU_WEEK,DOUBLE,YES,None,None,None
119,AMT_REQ_CREDIT_BUREAU_MON,DOUBLE,YES,None,None,None
120,AMT_REQ_CREDIT_BUREAU_QRT,DOUBLE,YES,None,None,None


In [2]:
shape = con.execute("""
    SELECT
      (SELECT COUNT(*) FROM app_train) AS n_rows,
      (SELECT COUNT(*) FROM information_schema.columns WHERE table_name='app_train') AS n_columns
""").df()
display(shape)

,n_rows,n_columns
0,307511,122


In [4]:
query_categorical_numerical = """
SELECT
   COUNT(*) FILTER (WHERE data_type IN (
      'TINYINT','SMALLINT','INTEGER','BIGINT',
      'UTINYINT','USMALLINT','UINTEGER','UBIGINT',
      'FLOAT','DOUBLE','DECIMAL'
   )) AS numeric_columns,
   COUNT(*) FILTER (WHERE data_type IN ('VARCHAR','CHAR','STRING')) AS categorical_columns,
   COUNT(*) AS total_columns
FROM information_schema.columns
WHERE table_name = 'app_train';
"""

df_col_types = con.execute(query_categorical_numerical).df()
display(df_col_types)

,numeric_columns,categorical_columns,total_columns
0,106,15,122


In [19]:
# Select only columns of type 'VARCHAR'
df_types = con.execute("DESCRIBE app_train").df()
categorical_cols = df_types[df_types['column_type'] == 'VARCHAR']['column_name'].tolist()

print(f"Categorical columns found: {len(categorical_cols)}")

# See how many distinct values each categorical column has
for col in categorical_cols:
    distinct_count = con.execute(f"SELECT COUNT(DISTINCT {col}) FROM app_train").fetchone()[0]
    print(f"{col}: {distinct_count} categories")

Categorical columns found: 15
NAME_CONTRACT_TYPE: 2 categories
CODE_GENDER: 3 categories
FLAG_OWN_CAR: 2 categories
FLAG_OWN_REALTY: 2 categories
NAME_TYPE_SUITE: 7 categories
NAME_INCOME_TYPE: 8 categories
NAME_EDUCATION_TYPE: 5 categories
NAME_FAMILY_STATUS: 6 categories
NAME_HOUSING_TYPE: 6 categories
OCCUPATION_TYPE: 18 categories
WEEKDAY_APPR_PROCESS_START: 7 categories
ORGANIZATION_TYPE: 58 categories
FONDKAPREMONT_MODE: 4 categories
HOUSETYPE_MODE: 3 categories
WALLSMATERIAL_MODE: 7 categories


In [3]:
# Check class balance for TARGET in app_train
target_balance_df_query = """
SELECT
    TARGET,
    COUNT(*) AS count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM app_train), 2) AS percentage
FROM app_train
GROUP BY TARGET
ORDER BY TARGET;
"""

target_balance_df = con.execute(target_balance_df_query).df()
display(target_balance_df)

,TARGET,count,percentage
0,0,282686,91.93
1,1,24825,8.07


In [8]:
# Check for missing values in app_train
import pandas as pd

# 1. Fetch only the schema (column names) without loading the data
schema_df = con.execute("DESCRIBE app_train").df()
columns = schema_df['column_name'].tolist()

# 2. Dynamically build a massive SQL query to calculate missing values for ALL columns
# The formula COUNT(*) - COUNT(column_name) finds the exact number of NULLs.
# We delegate this heavy aggregation entirely to DuckDB to prevent RAM overflow.
sql_select_statements = []
for col in columns:
    # We calculate the percentage directly in SQL
    statement = f"ROUND((COUNT(*) - COUNT({col})) * 100.0 / COUNT(*), 2) AS {col}_missing_pct"
    sql_select_statements.append(statement)

# Join all 122 statements into a single SELECT query
full_sql_query = "SELECT \n    " + ",\n    ".join(sql_select_statements) + "\nFROM app_train;"

# 3. Execute the aggregation query. 
# This returns a DataFrame with exactly 1 row and 122 columns, taking almost 0 RAM.
missing_stats_df = con.execute(full_sql_query).df()

# 4. Transpose the result to make it a vertical table, then clean and filter it
missing_stats = missing_stats_df.T.reset_index()
missing_stats.columns = ['Feature', 'Missing_Ratio_Percentage']

# Remove the '_missing_pct' suffix for cleaner reading
missing_stats['Feature'] = missing_stats['Feature'].str.replace('_missing_pct', '')

# Keep only features that actually have missing data (> 0%) and sort them
missing_stats = missing_stats[missing_stats['Missing_Ratio_Percentage'] > 0]
missing_stats = missing_stats.sort_values(by='Missing_Ratio_Percentage', ascending=False)

# 5. Display the top 20 features with the highest percentage of missing values
print(f"Total columns with missing data: {len(missing_stats)}")
display(missing_stats.head(20))

Total columns with missing data: 64


,Feature,Missing_Ratio_Percentage
48,COMMONAREA_AVG,69.87
62,COMMONAREA_MODE,69.87
76,COMMONAREA_MEDI,69.87
56,NONLIVINGAPARTMENTS_AVG,69.43
70,NONLIVINGAPARTMENTS_MODE,69.43
84,NONLIVINGAPARTMENTS_MEDI,69.43
86,FONDKAPREMONT_MODE,68.39
68,LIVINGAPARTMENTS_MODE,68.35
54,LIVINGAPARTMENTS_AVG,68.35
82,LIVINGAPARTMENTS_MEDI,68.35


### EDUCATION_TYPE

In [15]:
# See what types of values a column has
con.execute("SELECT DISTINCT NAME_EDUCATION_TYPE FROM app_train;").df()

,NAME_EDUCATION_TYPE
0,Incomplete higher
1,Academic degree
2,Secondary / secondary special
3,Higher education
4,Lower secondary


In [16]:
# See what types of values a column has and their counts
query_edu = """
SELECT 
    NAME_EDUCATION_TYPE, 
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM app_train), 2) AS percentage
FROM app_train
GROUP BY NAME_EDUCATION_TYPE
ORDER BY count DESC;
"""
display(con.execute(query_edu).df())

,NAME_EDUCATION_TYPE,count,percentage
0,Secondary / secondary special,218391,71.02
1,Higher education,74863,24.34
2,Incomplete higher,10277,3.34
3,Lower secondary,3816,1.24
4,Academic degree,164,0.05


In [20]:
def analyze_categorical(column_name):
    query = f"""
    SELECT 
        {column_name}, 
        COUNT(*) AS number_of_applicants,
        ROUND(AVG(TARGET), 2) AS default_rate,
    FROM app_train
    GROUP BY {column_name}
    ORDER BY default_rate DESC;
    """
    return con.execute(query).df()

# Who is more risky based on education type?
display(analyze_categorical('NAME_EDUCATION_TYPE'))

,NAME_EDUCATION_TYPE,number_of_applicants,default_rate
0,Lower secondary,3816,0.11
1,Secondary / secondary special,218391,0.09
2,Incomplete higher,10277,0.08
3,Higher education,74863,0.05
4,Academic degree,164,0.02


### EXT_SOURCE 

In [22]:
# Count total rows and missing values for specific columns
null_query = """
    SELECT 
        COUNT(*) AS total_applicants,
        COUNT(*) - COUNT(AMT_INCOME_TOTAL) AS missing_income,
        COUNT(*) - COUNT(EXT_SOURCE_1) AS missing_ext_source_1,
        COUNT(*) - COUNT(EXT_SOURCE_2) AS missing_ext_source_2,
        COUNT(*) - COUNT(EXT_SOURCE_3) AS missing_ext_source_3
    FROM app_train;
"""

# Execute and display the result
df_nulls = con.execute(null_query).df()
display(df_nulls)

,total_applicants,missing_income,missing_ext_source_1,missing_ext_source_2,missing_ext_source_3
0,307511,0,173378,660,60965


In [23]:
# Calculate default rates based on missing data
missingness_query = """
    SELECT
        CASE 
            WHEN EXT_SOURCE_1 IS NULL THEN 'Missing Score'
            ELSE 'Has Score'
        END AS ext_source_1_status,
        COUNT(*) AS total_applicants,
        AVG(TARGET) AS default_rate
    FROM app_train
    GROUP BY 
        ext_source_1_status;
"""

# Execute and display the result
df_missingness = con.execute(missingness_query).df()
display(df_missingness)

,ext_source_1_status,total_applicants,default_rate
0,Missing Score,173378,0.085195
1,Has Score,134133,0.074955


### DAYS_EMPLOYED

In [24]:
min_max_query = """
SELECT MIN(DAYS_EMPLOYED), MAX(DAYS_EMPLOYED) 
FROM app_train;
"""

df_min_max = con.execute(min_max_query).df()
display(df_min_max)

,min(DAYS_EMPLOYED),max(DAYS_EMPLOYED)
0,-17912,365243


In [25]:
income_type_query = """
    SELECT NAME_INCOME_TYPE, COUNT(*)
    FROM app_train
    WHERE DAYS_EMPLOYED = 365243
    GROUP BY NAME_INCOME_TYPE;
    """
df_income_type = con.execute(income_type_query).df()
display(df_income_type)

,NAME_INCOME_TYPE,count_star()
0,Unemployed,22
1,Pensioner,55352


In [26]:
impact_query = """
SELECT 
    CASE WHEN DAYS_EMPLOYED = 365243 THEN 'Anomalous' ELSE 'Regular' END AS status,
    COUNT(*) AS total_clients,
    AVG(TARGET) AS default_rate
FROM app_train
GROUP BY status;
"""

df_impact = con.execute(impact_query).df()
display(df_impact)

,status,total_clients,default_rate
0,Regular,252137,0.086600
1,Anomalous,55374,0.053996


In [27]:
# Query to check the cleaning of DAYS_EMPLOYED
query_days_clean = """
    SELECT 
        SK_ID_CURR,
        DAYS_EMPLOYED,
        CASE
            WHEN DAYS_EMPLOYED = 365243 THEN NULL 
            ELSE DAYS_EMPLOYED 
        END AS DAYS_EMPLOYED_CLEAN,
        CASE 
            WHEN DAYS_EMPLOYED = 365243 THEN TRUE 
            ELSE FALSE 
        END AS IS_DAYS_EMPLOYED_ANOM
    FROM app_train
    LIMIT 10;
"""

df_days_check = con.execute(query_days_clean).df()
display(df_days_check)

,SK_ID_CURR,DAYS_EMPLOYED,DAYS_EMPLOYED_CLEAN,IS_DAYS_EMPLOYED_ANOM
0,100002,-637,-637,False
1,100003,-1188,-1188,False
2,100004,-225,-225,False
3,100006,-3039,-3039,False
4,100007,-3038,-3038,False
5,100008,-1588,-1588,False
6,100009,-3130,-3130,False
7,100010,-449,-449,False
8,100011,365243,<NA>,True
9,100012,-2019,-2019,False


In [28]:
verification_query = """
SELECT 
    NAME_INCOME_TYPE, 
    COUNT(*) AS total,
    AVG(CASE WHEN DAYS_EMPLOYED = 365243 THEN 1 ELSE 0 END) AS ratio_anomali
FROM app_train
GROUP BY NAME_INCOME_TYPE;
"""

df_verification = con.execute(verification_query).df()
display(df_verification)

,NAME_INCOME_TYPE,total,ratio_anomali
0,Pensioner,55362,0.999819
1,State servant,21703,0.000000
2,Unemployed,22,1.000000
3,Working,158774,0.000000
4,Student,18,0.000000
5,Maternity leave,5,0.000000
6,Commercial associate,71617,0.000000
7,Businessman,10,0.000000


### OCCUPATION_TYPE

In [ ]:
occupation_query = """
SELECT
    OCCUPATION_TYPE,
    COUNT(*) AS total_applicants
FROM app_train
GROUP BY OCCUPATION_TYPE;
"""

occupation_df = con.execute(occupation_query).df()
display(occupation_df)

,OCCUPATION_TYPE,total_applicants
0,High skill tech staff,11380
1,Cooking staff,5946
2,Secretaries,1305
3,Managers,21371
4,Laborers,55186
5,Cleaning staff,4653
6,Medicine staff,8537
7,Accountants,9813
8,Drivers,18603
9,Security staff,6721


In [30]:
# Replace NaNs with 'Not specified'
query_occ = """
SELECT 
    COALESCE(OCCUPATION_TYPE, 'Not Specified') AS occupation,
    COUNT(*) AS count,
    AVG(TARGET) AS default_rate
FROM app_train
GROUP BY occupation
ORDER BY default_rate DESC;
"""
display(con.execute(query_occ).df())

,occupation,count,default_rate
0,Low-skill Laborers,2093,0.171524
1,Drivers,18603,0.113261
2,Waiters/barmen staff,1348,0.112760
3,Security staff,6721,0.107424
4,Laborers,55186,0.105788
5,Cooking staff,5946,0.104440
6,Sales staff,32102,0.096318
7,Cleaning staff,4653,0.096067
8,Realty agents,751,0.078562
9,Secretaries,1305,0.070498


In [31]:
# Is there a correlation between income type and missing occupation?
cross_check_query = """
SELECT 
    NAME_INCOME_TYPE,
    COUNT(*) AS total_per_category,
    COUNT(OCCUPATION_TYPE) AS with_occupation,
    COUNT(*) - COUNT(OCCUPATION_TYPE) AS without_occupation,
    ROUND((COUNT(*) - COUNT(OCCUPATION_TYPE)) * 100.0 / COUNT(*), 2) AS percentage_null
FROM app_train
GROUP BY NAME_INCOME_TYPE
ORDER BY without_occupation DESC;
"""

df_cross = con.execute(cross_check_query).df()
display(df_cross)

,NAME_INCOME_TYPE,total_per_category,with_occupation,without_occupation,percentage_null
0,Pensioner,55362,5,55357,99.99
1,Working,158774,133854,24920,15.70
2,Commercial associate,71617,59320,12297,17.17
3,State servant,21703,17916,3787,17.45
4,Unemployed,22,0,22,100.00
5,Student,18,13,5,27.78
6,Businessman,10,8,2,20.00
7,Maternity leave,5,4,1,20.00
